# 01 - Dataset Inspection and Cleaning: Online Retail II

**Project:** EconoCausal: Dynamic Pricing via Double Machine Learning
**Issue:** #1 (Dataset Collection)
**Author:** Duppada Dhanalakshmi

**Purpose.** Load the raw Online Retail II data, audit its quality, apply configurable and documented cleaning rules, and save a cleaned file for the team (EDA in Issue #2, DML methodology in Issue #3).

**Team decision (from review on Issue #1).** Quantity outliers (above 1,000 units) and high or unusual prices are **flagged, not removed**, until the EDA compares results with and without them. Every cleaning rule can be switched on or off in the configuration cell, and the number of rows affected by each rule is logged.

**Scope.** Inspection and cleaning only. Visual EDA and model work are separate tasks.

**Dataset.** Online Retail II, UCI Machine Learning Repository (CC BY 4.0). https://archive.ics.uci.edu/dataset/502/online+retail+ii

**How to run.** Place `online_retail_II.xlsx` in `data/raw/` (not committed to Git) and run all cells from the `notebooks/` folder. Needs `pandas`, `numpy`, `openpyxl`.

## 0. Configuration (all cleaning rules are set here)

Each `drop_*` switch turns a row-removal rule on or off. Thresholds for flags are set below the switches. Change a value and re-run the notebook to see the effect.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

RAW_PATH = Path("../data/raw/online_retail_II.xlsx")
OUT_PATH = Path("../data/processed/online_retail_clean.csv.gz")

# ---- Row-removal rules: True = remove the rows, False = keep them ----
CONFIG = {
    "drop_duplicates":           True,
    "drop_cancellations":        True,
    "drop_nonpositive_quantity": True,
    "drop_nonpositive_price":    True,
    "drop_non_product_codes":    True,
    "drop_quantity_outliers":    False,   # team decision: flag only, for now
    "drop_price_outliers":       False,   # team decision: flag only, for now
}

# ---- Flag thresholds (used for flags even when the rule above is False) ----
QUANTITY_OUTLIER_THRESHOLD = 1000   # Quantity above this is flagged as a bulk-order outlier
PRICE_ABS_THRESHOLD        = 100    # Price above this (GBP) is flagged as high price
PRICE_RATIO_THRESHOLD      = 5      # Price more than 5x above or below the product's median is flagged
PRICE_RATIO_MIN_LINES      = 5      # ... only for products with at least this many lines

# Stock codes that are not products (postage, fees, manual adjustments, vouchers, tests)
NON_PRODUCT_CODES = {
    "POST", "DOT", "M", "D", "S", "B", "C2", "ADJUST", "ADJUST2",
    "BANK CHARGES", "PADS", "CRUK", "AMAZONFEE", "TEST001", "TEST002",
}
NON_PRODUCT_PREFIXES = ("GIFT_",)

assert RAW_PATH.exists(), f"Put the dataset at {RAW_PATH.resolve()}"
print(pd.Series(CONFIG, name="enabled").to_string())

drop_duplicates               True
drop_cancellations            True
drop_nonpositive_quantity     True
drop_nonpositive_price        True
drop_non_product_codes        True
drop_quantity_outliers       False
drop_price_outliers          False

## 1. Load the data

The Excel file has two sheets (one per year). They are combined into one table, keeping the sheet name so rows can be traced back.

In [2]:
xl = pd.ExcelFile(RAW_PATH)
frames = []
for sheet in xl.sheet_names:
    part = xl.parse(sheet)
    part["source_sheet"] = sheet
    print(f"{sheet}: {part.shape[0]:,} rows x {part.shape[1] - 1} columns")
    frames.append(part)

raw = pd.concat(frames, ignore_index=True)
print(f"\nCombined: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

Year 2009-2010: 525,461 rows x 8 columns
Year 2010-2011: 541,910 rows x 8 columns

Combined: 1,067,371 rows x 9 columns

## 2. Initial inspection

In [3]:
print(raw.dtypes.to_string())
print()
print(raw.drop(columns="source_sheet").head().to_string())
print()
print("Date range:", raw["InvoiceDate"].min(), "to", raw["InvoiceDate"].max())
print()
print("Unique invoices :", f"{raw['Invoice'].nunique():,}")
print("Unique products :", f"{raw['StockCode'].astype(str).nunique():,}  (raw StockCode, before normalising case)")
print("Unique customers:", f"{raw['Customer ID'].nunique():,}")
print("Countries       :", raw["Country"].nunique())
print()
print(raw[["Quantity", "Price"]].describe(percentiles=[.01, .5, .99]).round(2).to_string())

Invoice                 object
StockCode               object
Description             object
Quantity                 int64
InvoiceDate     datetime64[us]
Price                  float64
Customer ID            float64
Country                    str
source_sheet               str

  Invoice StockCode                          Description  Quantity         InvoiceDate  Price  Customer ID         Country
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12 2009-12-01 07:45:00   6.95      13085.0  United Kingdom
1  489434    79323P                   PINK CHERRY LIGHTS        12 2009-12-01 07:45:00   6.75      13085.0  United Kingdom
2  489434    79323W                  WHITE CHERRY LIGHTS        12 2009-12-01 07:45:00   6.75      13085.0  United Kingdom
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48 2009-12-01 07:45:00   2.10      13085.0  United Kingdom
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24 2009-12-01 07:45:00   1.25      13085.

Note: the Excel file uses `Invoice`, `Price` and `Customer ID`, while the UCI web page calls them InvoiceNo, UnitPrice and CustomerID.

## 3. Data quality audit (raw data, each issue counted independently)

The categories overlap (for example, a cancellation row also has negative quantity), so these counts should not be added together. Section 4 gives the sequential counts for each rule.

In [4]:
n = len(raw)
inv = raw["Invoice"].astype(str).str.strip().str.upper()
code_ = raw["StockCode"].astype(str).str.strip().str.upper()
is_non_product = code_.isin(NON_PRODUCT_CODES) | code_.str.startswith(NON_PRODUCT_PREFIXES)

audit = pd.DataFrame({
    "issue": [
        "Missing Customer ID",
        "Missing Description",
        "Fully duplicated rows (excluding source_sheet)",
        "Cancellation invoices (prefix C)",
        "Negative Quantity",
        "Zero Price",
        "Negative Price",
        "Non-product stock codes",
        f"Quantity above {QUANTITY_OUTLIER_THRESHOLD:,}",
        f"Price above {PRICE_ABS_THRESHOLD}",
    ],
    "rows": [
        raw["Customer ID"].isna().sum(),
        raw["Description"].isna().sum(),
        raw.drop(columns="source_sheet").duplicated().sum(),
        inv.str.startswith("C").sum(),
        (raw["Quantity"] < 0).sum(),
        (raw["Price"] == 0).sum(),
        (raw["Price"] < 0).sum(),
        is_non_product.sum(),
        (raw["Quantity"] > QUANTITY_OUTLIER_THRESHOLD).sum(),
        (raw["Price"] > PRICE_ABS_THRESHOLD).sum(),
    ],
})
audit["pct_of_rows"] = (audit["rows"] / n * 100).round(2)
print(audit.to_string(index=False))

# Product codes that only differ by letter case would be counted as different products
print()
print("StockCode count as-is        :", f"{raw['StockCode'].astype(str).nunique():,}")
print("StockCode count after .upper():", f"{code_.nunique():,}")

                                         issue   rows  pct_of_rows
                           Missing Customer ID 243007        22.77
                           Missing Description   4382         0.41
Fully duplicated rows (excluding source_sheet)  34335         3.22
              Cancellation invoices (prefix C)  19494         1.83
                             Negative Quantity  22950         2.15
                                    Zero Price   6202         0.58
                                Negative Price      5         0.00
                       Non-product stock codes   5930         0.56
                          Quantity above 1,000    352         0.03
                               Price above 100   1946         0.18

StockCode count as-is        : 5,305
StockCode count after .upper(): 5,131

## 4. Apply the cleaning rules

Rules are applied in the order below. For each rule the log records:
- `rows_matching`: rows that meet the rule's condition at that step
- `rows_removed`: rows actually removed (zero when the rule is switched off)
- `rows_remaining`: rows left afterwards

Because rules run in order, `rows_matching` can be smaller than the independent counts in section 3 (a row already removed by an earlier rule is not counted again).

| Rule | What it matches | Why |
|---|---|---|
| Standardise identifiers | (no rows removed) strip spaces and upper-case `Invoice` and `StockCode` | The same product can appear with different letter case, which would split its price history |
| `drop_duplicates` | Fully duplicated rows | Repeated lines double-count demand |
| `drop_cancellations` | Invoices starting with C | Returns are not demand at the posted price |
| `drop_nonpositive_quantity` | Quantity <= 0 | Not a sale |
| `drop_nonpositive_price` | Price <= 0 | Free items and bad-debt adjustments are not market prices |
| `drop_non_product_codes` | Postage, fees, adjustments, vouchers, tests | These do not respond to price like products |
| `drop_quantity_outliers` | Quantity above the threshold | Bulk orders can dominate a quantity outcome. **Currently flagged only** |
| `drop_price_outliers` | Price above the absolute threshold, or far from the product's median | May be real high-value items or entry errors. **Currently flagged only** |

`Customer ID` is **not imputed**. About a quarter of rows have none, so it is kept only as a `known_customer` flag.

In [5]:
log = []

def apply_rule(df, name, rule_key, bad_mask, reason):
    # Count rows matching the rule, and remove them only if the rule is enabled.
    matching = int(bad_mask.sum())
    before = len(df)
    if CONFIG.get(rule_key, False):
        df = df[~bad_mask]
    log.append({
        "rule": name,
        "removing": CONFIG.get(rule_key, False),
        "rows_matching": matching,
        "rows_removed": before - len(df),
        "rows_remaining": len(df),
        "reason": reason,
    })
    return df

df = raw.copy()

# Standardise identifiers (no rows removed)
df["Invoice"] = df["Invoice"].astype(str).str.strip().str.upper()
df["StockCode"] = df["StockCode"].astype(str).str.strip().str.upper()

# Rule 1: duplicates (compare all original columns, ignore the helper column)
cols = [c for c in df.columns if c != "source_sheet"]
df = apply_rule(df, "duplicates", "drop_duplicates",
                df.duplicated(subset=cols), "Repeated lines double-count demand")

# Rule 2: cancellations
df = apply_rule(df, "cancellation invoices", "drop_cancellations",
                df["Invoice"].str.startswith("C"), "Returns are not demand")

# Rule 3: non-positive quantity
df = apply_rule(df, "Quantity <= 0", "drop_nonpositive_quantity",
                df["Quantity"] <= 0, "Not a sale")

# Rule 4: non-positive price
df = apply_rule(df, "Price <= 0", "drop_nonpositive_price",
                df["Price"] <= 0, "Free items / adjustments are not market prices")

# Rule 5: non-product stock codes
is_np = df["StockCode"].isin(NON_PRODUCT_CODES) | df["StockCode"].str.startswith(NON_PRODUCT_PREFIXES)
df = apply_rule(df, "non-product stock codes", "drop_non_product_codes",
                is_np, "Postage, fees, vouchers, tests")

# Flags for rules 6 and 7 are always created, so the rows can be studied even when kept.
df = df.reset_index(drop=True)
df["quantity_outlier"] = df["Quantity"] > QUANTITY_OUTLIER_THRESHOLD

df["product_median_price"] = df.groupby("StockCode")["Price"].transform("median")
df["product_lines"] = df.groupby("StockCode")["Price"].transform("size")
ratio = df["Price"] / df["product_median_price"]
df["price_high_abs"] = df["Price"] > PRICE_ABS_THRESHOLD
df["price_ratio_outlier"] = (
    (df["product_lines"] >= PRICE_RATIO_MIN_LINES)
    & ((ratio > PRICE_RATIO_THRESHOLD) | (ratio < 1 / PRICE_RATIO_THRESHOLD))
)
df["price_outlier"] = df["price_high_abs"] | df["price_ratio_outlier"]

# Rule 6: quantity outliers (flag only unless switched on)
df = apply_rule(df, f"Quantity > {QUANTITY_OUTLIER_THRESHOLD:,} (flag: quantity_outlier)",
                "drop_quantity_outliers", df["quantity_outlier"], "Bulk orders, kept for now")

# Rule 7: price outliers (flag only unless switched on)
df = apply_rule(df, "price outliers (flag: price_outlier)",
                "drop_price_outliers", df["price_outlier"], "High or unusual prices, kept for now")

clean = df.copy()
print("Rows remaining:", f"{len(clean):,}")

Rows remaining: 1,003,340

In [6]:
log_df = pd.DataFrame(log)
print(log_df[["rule", "removing", "rows_matching", "rows_removed", "rows_remaining"]].to_string(index=False))
print()
total_removed = len(raw) - len(clean)
print(f"Raw rows   : {len(raw):,}")
print(f"Clean rows : {len(clean):,}")
print(f"Removed    : {total_removed:,} ({total_removed / len(raw) * 100:.1f}%)")
print(f"Kept       : {len(clean) / len(raw) * 100:.1f}%")
print()
print("Flagged rows kept in the data:")
print(f"  quantity_outlier   : {clean['quantity_outlier'].sum():,}")
print(f"  price_high_abs     : {clean['price_high_abs'].sum():,}")
print(f"  price_ratio_outlier: {clean['price_ratio_outlier'].sum():,}")
print(f"  price_outlier (any): {clean['price_outlier'].sum():,}")

                                     rule  removing  rows_matching  rows_removed  rows_remaining
                               duplicates      True          34335         34335         1033036
                    cancellation invoices      True          19104         19104         1013932
                            Quantity <= 0      True           3393          3393         1010539
                               Price <= 0      True           2626          2626         1007913
                  non-product stock codes      True           4573          4573         1003340
Quantity > 1,000 (flag: quantity_outlier)     False            299             0         1003340
     price outliers (flag: price_outlier)     False           3328             0         1003340

Raw rows   : 1,067,371
Clean rows : 1,003,340
Removed    : 64,031 (6.0%)
Kept       : 94.0%

Flagged rows kept in the data:
  quantity_outlier   : 299
  price_high_abs     : 187
  price_ratio_outlier: 3,149
  price_outlier 

## 5. Light feature preparation

Only columns that follow directly from the cleaning rules are added. Modelling features (lags, holidays, aggregation) are left to the methodology work.

- `known_customer`: whether a Customer ID exists
- `log_price`, `log_quantity`: natural logs, so a regression of one on the other reads as a price elasticity
- `week`: Monday that starts the invoice week, and `month`

In [7]:
clean["known_customer"] = clean["Customer ID"].notna()
clean["Customer ID"] = clean["Customer ID"].astype("Int64")
clean["revenue"] = clean["Quantity"] * clean["Price"]
clean["log_price"] = np.log(clean["Price"])
clean["log_quantity"] = np.log(clean["Quantity"])
clean["week"] = clean["InvoiceDate"].dt.to_period("W-SUN").dt.start_time
clean["month"] = clean["InvoiceDate"].dt.to_period("M").astype(str)
clean = clean.reset_index(drop=True)

print(clean.dtypes.to_string())

Invoice                            str
StockCode                          str
Description                     object
Quantity                         int64
InvoiceDate             datetime64[us]
Price                          float64
Customer ID                      Int64
Country                            str
source_sheet                       str
quantity_outlier                  bool
product_median_price           float64
product_lines                    int64
price_high_abs                    bool
price_ratio_outlier               bool
price_outlier                     bool
known_customer                    bool
revenue                        float64
log_price                      float64
log_quantity                   float64
week                    datetime64[us]
month                              str

## 6. Characteristics of the flagged quantity outliers

Requested in review: document what the rows with Quantity above the threshold look like, so the final cutoff can be justified.

In [8]:
qo = clean[clean["quantity_outlier"]]
print(f"Threshold: Quantity > {QUANTITY_OUTLIER_THRESHOLD:,}")
print(f"Rows               : {len(qo):,} ({len(qo) / len(clean) * 100:.3f}% of cleaned rows)")
print(f"Share of all units : {qo['Quantity'].sum() / clean['Quantity'].sum() * 100:.1f}%")
print(f"Share of revenue   : {qo['revenue'].sum() / clean['revenue'].sum() * 100:.1f}%")
print(f"Invoices           : {qo['Invoice'].nunique():,}")
print(f"Products           : {qo['StockCode'].nunique():,}")
print(f"Customers (known)  : {qo['Customer ID'].nunique():,}")
print(f"Rows with no Customer ID: {(~qo['known_customer']).mean() * 100:.1f}% (all rows: {(~clean['known_customer']).mean() * 100:.1f}%)")
print()
print("Quantity of flagged rows:")
print(qo["Quantity"].describe().round(0).to_string())
print()
print("Price of flagged rows vs all rows (median):",
      round(qo["Price"].median(), 2), "vs", round(clean["Price"].median(), 2))
print()
print("Countries:")
print(qo["Country"].value_counts().head(5).to_string())
print()
print("Most frequent products:")
print(qo["Description"].value_counts().head(8).to_string())
print()
print("Invoices with the most flagged lines:")
print(qo.groupby("Invoice").size().sort_values(ascending=False).head(5).to_string())
print()
print("Flagged rows by month:")
print(qo.groupby("month").size().to_string())

Threshold: Quantity > 1,000
Rows               : 299 (0.030% of cleaned rows)
Share of all units : 8.3%
Share of revenue   : 2.9%
Invoices           : 158
Products           : 155
Customers (known)  : 61
Rows with no Customer ID: 2.0% (all rows: 22.6%)

Quantity of flagged rows:
count      299.0
mean      3120.0
std       6569.0
min       1002.0
25%       1200.0
50%       1584.0
75%       3056.0
max      80995.0

Price of flagged rows vs all rows (median): 0.25 vs 2.1

Countries:
Country
United Kingdom    216
Denmark            41
France             25
Sweden              5
Netherlands         5

Most frequent products:
Description
WORLD WAR 2 GLIDERS ASSTD DESIGNS    20
GIRLS ALPHABET IRON ON PATCHES       10
SMALL CHINESE STYLE SCISSOR           8
BROCADE RING PURSE                    7
RED  HARMONICA IN BOX                 7
MINI HIGHLIGHTER PENS                 6
POPCORN HOLDER                        6
SMALL POPCORN HOLDER                  5

Invoices with the most flagged lines:
I

## 7. High and unusual prices (flagged, kept)

Requested in review: retain high-priced items and document any unusual price values. Two flags are used:
- `price_high_abs`: Price above the absolute threshold
- `price_ratio_outlier`: Price more than 5x above or below that product's own median (for products with enough lines)

In [9]:
pa = clean[clean["price_high_abs"]]
print(f"Price above {PRICE_ABS_THRESHOLD}: {len(pa):,} rows, {pa['StockCode'].nunique()} products")
print()
print("Most frequent high-price products (with their median price):")
top = (pa.groupby("Description")
         .agg(rows=("Price", "size"), median_price=("Price", "median"),
              min_price=("Price", "min"), max_price=("Price", "max"))
         .sort_values("rows", ascending=False).head(10))
print(top.round(2).to_string())

pr = clean[clean["price_ratio_outlier"]]
print()
print(f"Price more than {PRICE_RATIO_THRESHOLD}x above/below product median: {len(pr):,} rows, {pr['StockCode'].nunique():,} products")
print(f"  of which above the median: {(pr['Price'] > pr['product_median_price']).sum():,}")
print(f"  of which below the median: {(pr['Price'] < pr['product_median_price']).sum():,}")
print()
print("Highest prices overall, with the product's typical (median) price:")
cols = ["StockCode", "Description", "Quantity", "Price", "product_median_price", "Country"]
print(clean.sort_values("Price", ascending=False).head(12)[cols].round(2).to_string(index=False))
print()
print("Largest price-to-median ratios among flagged rows:")
tmp = pr.assign(ratio=pr["Price"] / pr["product_median_price"]).sort_values("ratio", ascending=False)
print(tmp.head(10)[cols + ["ratio"]].round(2).to_string(index=False))

Price above 100: 187 rows, 13 products

Most frequent high-price products (with their median price):
                                     rows  median_price  min_price  max_price
Description                                                                  
VINTAGE RED KITCHEN CABINET            51         125.0     125.00     295.00
VINTAGE BLUE KITCHEN CABINET           36         295.0     125.00     295.00
RUSTIC  SEVENTEEN DRAWER SIDEBOARD     29         165.0     145.00     165.00
CHEST NATURAL WOOD 20 DRAWERS          18         117.5     110.00     125.00
FRENCH STYLE WALL DRESSER              12         125.0     110.00     125.00
LOVE SEAT ANTIQUE WHITE METAL          11         195.0     175.00     195.00
REGENCY MIRROR WITH SHUTTERS           11         145.0     145.00     165.00
GIANT SEVENTEEN DRAWER SIDEBOARD        6         165.0     145.00     165.00
FLAG OF ST GEORGE CAR FLAG              5         408.4     272.27    1157.15
ANT WHITE SWEETHEART TABLE W 3 DRAW     2

**How to read this.** High prices are not all alike. Some belong to genuinely expensive products (the furniture items). Others are low-cost products with a one-off very high price, which looks like a data-entry error. The tables above separate the two by showing the product's typical price next to each unusual price. The final handling (keep, cap, or remove) is left for the team to decide after the EDA.

## 8. With versus without the flagged rows

Requested in review: compare results with and without the flagged rows, to help justify the final cutoff. This is a quick descriptive comparison, not the DML estimate.

The "naive slope" below is a within-product regression of log weekly quantity on log weekly average price. It ignores confounding (promotions, seasonality, order mix), so it is not a causal elasticity. It is only used to see how sensitive a simple estimate is to the flagged rows.

In [10]:
def summarize(d):
    pw = (d.groupby(["StockCode", "week"])
            .agg(avg_price=("Price", "mean"), qty=("Quantity", "sum"))
            .reset_index())
    lp = np.log(pw["avg_price"])
    lq = np.log(pw["qty"])
    lp_w = lp - lp.groupby(pw["StockCode"]).transform("mean")
    lq_w = lq - lq.groupby(pw["StockCode"]).transform("mean")
    slope = float((lp_w * lq_w).sum() / (lp_w ** 2).sum())
    return {
        "rows": len(d),
        "total_units": int(d["Quantity"].sum()),
        "total_revenue_GBP": round(float(d["revenue"].sum())),
        "mean_quantity": round(float(d["Quantity"].mean()), 2),
        "median_quantity": float(d["Quantity"].median()),
        "product_week_rows": len(pw),
        "naive_within_product_slope": round(slope, 3),
    }

variants = {
    "all rows (current default)": clean,
    "without quantity outliers": clean[~clean["quantity_outlier"]],
    "without price outliers": clean[~clean["price_outlier"]],
    "without both": clean[~clean["quantity_outlier"] & ~clean["price_outlier"]],
}
comparison = pd.DataFrame({k: summarize(v) for k, v in variants.items()}).T
print(comparison.to_string())

                                 rows  total_units  total_revenue_GBP  mean_quantity  median_quantity  product_week_rows  naive_within_product_slope
all rows (current default)  1003340.0   11188044.0         19643862.0          11.15              4.0           196242.0                      -1.390
without quantity outliers   1003041.0   10255043.0         19064833.0          10.22              4.0           196174.0                      -1.366
without price outliers      1000012.0   10805214.0         19473866.0          10.81              4.0           195287.0                      -1.508
without both                 999797.0   10198189.0         18937226.0          10.20              4.0           195262.0                      -1.503

## 9. Validation checks

These assertions fail loudly if the cleaning did not do what the log says.

In [11]:
if CONFIG["drop_nonpositive_quantity"]:
    assert clean["Quantity"].gt(0).all()
if CONFIG["drop_nonpositive_price"]:
    assert clean["Price"].gt(0).all()
if CONFIG["drop_cancellations"]:
    assert not clean["Invoice"].str.startswith("C").any()
if CONFIG["drop_non_product_codes"]:
    assert not clean["StockCode"].isin(NON_PRODUCT_CODES).any()
    assert not clean["StockCode"].str.startswith(NON_PRODUCT_PREFIXES).any()
if CONFIG["drop_quantity_outliers"]:
    assert not clean["quantity_outlier"].any()
if CONFIG["drop_duplicates"]:
    key = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]
    assert not clean.duplicated(subset=key).any()
assert clean[["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Country"]].notna().all().all()
assert clean["quantity_outlier"].sum() == (clean["Quantity"] > QUANTITY_OUTLIER_THRESHOLD).sum()
print("All validation checks passed.")
print()
print("Missing values in cleaned data:")
miss = clean.isna().sum()
print(miss[miss > 0].to_string() if (miss > 0).any() else "none")
print()
print(f"Customer ID missing: {(~clean['known_customer']).mean() * 100:.1f}% of cleaned rows")

All validation checks passed.

Missing values in cleaned data:
Customer ID    226761

Customer ID missing: 22.6% of cleaned rows

## 10. Checks relevant to the DML setup

A quick look at whether the cleaned data has the price variation DML needs. This is a rough check, and the full analysis belongs in EDA (Issue #2).

Rows are aggregated to **product-week** level (average price, total quantity), which is the planned modelling grain.

In [12]:
pw = (clean.groupby(["StockCode", "week"])
           .agg(avg_price=("Price", "mean"),
                total_qty=("Quantity", "sum"),
                n_lines=("Invoice", "size"))
           .reset_index())

print(f"Product-week rows : {len(pw):,}")
print(f"Products          : {pw['StockCode'].nunique():,}")
print(f"Weeks covered     : {pw['week'].nunique()}")

per_prod = pw.groupby("StockCode").agg(
    weeks=("week", "nunique"),
    distinct_prices=("avg_price", lambda s: s.round(2).nunique()),
    price_cv=("avg_price", lambda s: s.std() / s.mean() if len(s) > 1 else np.nan),
)

line_prices = clean.groupby("StockCode")["Price"].nunique()
print()
print(f"Products with more than one line-level price: {(line_prices > 1).mean() * 100:.1f}%")

enough = per_prod[per_prod["weeks"] >= 20]
print(f"Products with at least 20 weeks of data      : {len(enough):,} ({len(enough) / len(per_prod) * 100:.1f}%)")
print(f"  of these, share with >1 weekly price level : {(enough['distinct_prices'] > 1).mean() * 100:.1f}%")
print(f"  median CV of weekly price within product   : {enough['price_cv'].median():.3f}")
print(f"  share with weekly price CV above 5%        : {(enough['price_cv'] > 0.05).mean() * 100:.1f}%")

print()
print("Country share of cleaned rows (top 3):")
print((clean["Country"].value_counts(normalize=True).head(3) * 100).round(1).to_string())

Product-week rows : 196,242
Products          : 4,724
Weeks covered     : 104

Products with more than one line-level price: 89.9%
Products with at least 20 weeks of data      : 3,178 (67.3%)
  of these, share with >1 weekly price level : 99.1%
  median CV of weekly price within product   : 0.241
  share with weekly price CV above 5%        : 92.7%

Country share of cleaned rows (top 3):
Country
United Kingdom    92.0
EIRE               1.7
Germany            1.6

**Reading these numbers with care.** The median weekly price CV is high for a retailer. The average price of a product-week mixes wholesale and retail lines, so part of this variation comes from the order mix and not from real price changes. In EDA, check this with a mode or median price, or by restricting to UK retail-sized orders, before treating it as usable treatment variation.

## 11. Save the cleaned data

The cleaned file goes in `data/processed/` and should stay out of Git (add it to `.gitignore`). It includes the flag columns, so the team can filter flagged rows in EDA without re-running the cleaning. Anyone can regenerate it by running this notebook.

In [13]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
to_save = clean.drop(columns=["source_sheet", "product_lines"])
to_save.to_csv(OUT_PATH, index=False, compression="gzip")
size_mb = OUT_PATH.stat().st_size / 1e6
print(f"Saved {len(to_save):,} rows x {to_save.shape[1]} columns to {OUT_PATH} ({size_mb:.1f} MB)")
print("Flag columns:", [c for c in to_save.columns if c in ("quantity_outlier", "price_high_abs", "price_ratio_outlier", "price_outlier")])

check = pd.read_csv(OUT_PATH, parse_dates=["InvoiceDate", "week"])
assert len(check) == len(to_save)
print("Reload check passed.")

Saved 1,003,340 rows x 19 columns to ../data/processed/online_retail_clean.csv.gz (23.1 MB)
Flag columns: ['quantity_outlier', 'price_high_abs', 'price_ratio_outlier', 'price_outlier']
Reload check passed.

## 12. Summary of decisions and limitations

**Cleaned (rows removed).** Duplicates, cancellations, non-positive quantity and price, and non-product stock codes (counts in the log in section 4). Identifiers were upper-cased so the same product is not split by letter case.

**Flagged, not removed (team decision, pending EDA).**
- `quantity_outlier`: Quantity above the threshold. Characteristics are documented in section 6.
- `price_high_abs` and `price_ratio_outlier` (combined in `price_outlier`): high or unusual prices. Examples are documented in section 7. Some are genuine expensive items and some look like entry errors.
- Section 8 compares summary figures with and without these rows.

**Left alone on purpose.**
- Missing `Customer ID` was not imputed (kept as `known_customer` flag).
- Wholesale customers were not separated out, since the data has no customer type column.

**Configurable.** Every rule and threshold is set in the configuration cell, so the team can change a decision and re-run.

**Limitations that remain**
1. **No promotion or competitor-price data.** Price changes that coincide with promotions are an unobserved confounder, so elasticity estimates may be biased.
2. **Prices were not randomised.** Variation may come from wholesale versus retail pricing or product lifecycle, not pricing decisions.
3. **Quantity is not the same as demand.** There is no stock data, so low sales may reflect stockouts.
4. **Single retailer, mostly UK.** Results may not carry over to other markets.
5. **Customer-level analysis is limited** by the missing Customer ID values.
6. **Outlier thresholds are judgement calls.** They stay flagged until the EDA justifies a cutoff.

**Handover.** The cleaned file is ready for EDA (Issue #2). The team should decide on the outlier thresholds after the with/without comparison and before finalising the DML methodology (Issue #3).

*Data source: Chen, D. (2012). Online Retail II [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D (CC BY 4.0)*